In [1]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.stats.proportion as proportions_ztest
print("라이브러리 불러오기 완료")

라이브러리 불러오기 완료


In [2]:
import os

print(os.listdir("./data"))

['결제 정보 데이터.csv', '고객 센터 문의 데이터.csv', '구독 정보 데이터.csv', '유저 정보 데이터.csv', '일자별 행동 데이터.csv']


In [ ]:
# 데이터 불러오기
pay = pd.read_csv("./data/결제 정보 데이터.csv")

# 각 데이터셋의 데이터 확인

## 결제 정보 데이터

In [4]:
pay.head()

,payment_date,user_id,subscription_type,amount,payment_status
0,2026-01-15,U2501,yearly,294,성공
1,2026-01-21,U2503,yearly,294,성공
2,2026-01-20,U2504,monthly,35,성공
3,2026-01-27,U2505,monthly,35,성공
4,2026-01-22,U2506,monthly,35,성공


In [11]:
pay.info()

<class 'pandas.DataFrame'>
RangeIndex: 940 entries, 0 to 939
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   payment_date       940 non-null    str  
 1   user_id            940 non-null    str  
 2   subscription_type  940 non-null    str  
 3   amount             940 non-null    int64
 4   payment_status     940 non-null    str  
dtypes: int64(1), str(4)
memory usage: 36.8 KB


In [5]:
pay.isna().sum()

payment_date         0
user_id              0
subscription_type    0
amount               0
payment_status       0
dtype: int64

In [6]:
pay.duplicated().sum()

np.int64(8)

In [7]:
pay[pay.duplicated()]

,payment_date,user_id,subscription_type,amount,payment_status
29,2026-01-17,U2590,monthly,35,실패
81,2026-01-19,U2745,monthly,35,실패
111,2026-01-17,U2831,monthly,35,실패
161,2026-01-27,U2961,monthly,35,실패
367,2026-01-21,U3473,yearly,294,실패
606,2026-01-18,U4104,yearly,294,실패
852,2026-01-18,U4759,monthly,35,실패
876,2026-01-17,U4823,yearly,294,실패


In [12]:
# 중복된 데이터만 모아서 보기
pay[pay.duplicated(keep=False)].sort_values(by='user_id')

,payment_date,user_id,subscription_type,amount,payment_status
28,2026-01-17,U2590,monthly,35,실패
29,2026-01-17,U2590,monthly,35,실패
80,2026-01-19,U2745,monthly,35,실패
81,2026-01-19,U2745,monthly,35,실패
110,2026-01-17,U2831,monthly,35,실패
111,2026-01-17,U2831,monthly,35,실패
160,2026-01-27,U2961,monthly,35,실패
161,2026-01-27,U2961,monthly,35,실패
366,2026-01-21,U3473,yearly,294,실패
367,2026-01-21,U3473,yearly,294,실패


In [13]:
# 완전히 똑같은 중복 행은 1건만 남기고 제거 (원본 데이터에 바로 반영)
pay.drop_duplicates(inplace=True)

# 제거 후 데이터 개수(행 수) 다시 확인
print("중복 제거 후 데이터 크기:", pay.shape)

중복 제거 후 데이터 크기: (932, 5)


In [14]:
pay.duplicated().sum()

np.int64(0)

In [16]:
# amount 컬럼의 요약 통계량 확인
pay['amount'].describe()

count    932.000000
mean     197.847639
std      125.199855
min       35.000000
25%       35.000000
50%      294.000000
75%      294.000000
max      294.000000
Name: amount, dtype: float64

- 940개 데이터중 8개의 데이터의 중복이 확인되어 확인 후 중복 제거를 진행
- 결측이나, 이상치로 보이는것은 없었다.

In [ ]:
# payment_date 컬럼의 데이터 타입 확인
pay['payment_date'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [19]:
pay['payment_date'] = pd.to_datetime(pay['payment_date'])
print(pay['payment_date'].dtypes)

datetime64[us]


In [20]:
# 데이터의 가장 첫 날짜와 가장 마지막 날짜 확인
print("가장 빠른 결제일:", pay['payment_date'].min())
print("가장 늦은 결제일:", pay['payment_date'].max())

가장 빠른 결제일: 2026-01-15 00:00:00
가장 늦은 결제일: 2026-01-28 00:00:00


In [21]:
pay_success = pay[pay['payment_status'] == '성공']

print(pay_success['payment_status'].value_counts())

payment_status
성공    913
Name: count, dtype: int64


In [22]:
start_date = '2026-01-15'
end_date = '2026-01-28'

pay_final = pay_success[(pay_success['payment_date'] >= start_date) & (pay_success['payment_date'] <= end_date)]

print("필터링 후 시작일 : ", pay_final['payment_date'].min())
print("필터링 후 종료일 : ", pay_final['payment_date'].max())
print("최종 남은 데이터 건수 : ", pay_final.shape[0])

필터링 후 시작일 :  2026-01-15 00:00:00
필터링 후 종료일 :  2026-01-28 00:00:00
최종 남은 데이터 건수 :  913


In [ ]:
pay_user_level = pay_final.groupby('user_id').agg(
    total_amount=('amount', 'sum'),
    monthly_cnt=('subscription_type', lambda x: (x == 'monthly').sum()),
    yearly_cnt=('subscription_type', lambda x: (x == 'yearly').sum())
).reset_index()

,user_id,total_amount,monthly_cnt,yearly_cnt
0,U2501,294,0,1
1,U2503,294,0,1
2,U2504,35,1,0
3,U2505,35,1,0
4,U2506,35,1,0


In [26]:
pay_user_level['is_purchased'] = 1

print(pay_user_level.head())

  user_id  total_amount  monthly_cnt  yearly_cnt  is_purchased
0   U2501           294            0           1             1
1   U2503           294            0           1             1
2   U2504            35            1           0             1
3   U2505            35            1           0             1
4   U2506            35            1           0             1


# 유저 정보 데이터

In [ ]:
customer = pd.read_csv("./data/유저 정보 데이터.csv")

In [28]:
customer.head()

,user_id,user_type,group,gender,age
0,U2501,신규 고객,B,여성,24
1,U2502,신규 고객,A,여성,37
2,U2503,신규 고객,B,여성,30
3,U2504,신규 고객,A,남성,51
4,U2505,신규 고객,A,여성,37


## 결측치,중복값,이상치 확인

In [30]:
customer.isna().sum()

user_id      0
user_type    0
group        0
gender       0
age          0
dtype: int64

In [31]:
customer.duplicated().sum()

np.int64(0)

In [35]:
customer['group'].value_counts()

group
A    1277
B    1223
Name: count, dtype: int64

In [36]:
# 1. user_type 컬럼에 어떤 값들이 있는지 확인 (신규 고객만 있는지 검증)
print("유저 유형 분포:")
print(customer['user_type'].value_counts(dropna=False))
print("-" * 50)

# 2. gender(성별) 컬럼 고유값 확인
print("성별 분포:")
print(customer['gender'].value_counts(dropna=False))
print("-" * 50)

# 3. age(나이) 컬럼의 기초 통계량 확인 (말도 안 되는 나이가 있는지 검증)
print("나이 통계량:")
print(customer['age'].describe())

유저 유형 분포:
user_type
신규 고객    2500
Name: count, dtype: int64
--------------------------------------------------
성별 분포:
gender
남성    1271
여성    1229
Name: count, dtype: int64
--------------------------------------------------
나이 통계량:
count    2500.000000
mean       34.723600
std         9.076792
min        20.000000
25%        28.000000
50%        35.000000
75%        41.000000
max        60.000000
Name: age, dtype: float64


<> 유저 정보 데이터 끝

# 일자별 행동 데이터

In [37]:
dailybehavior = pd.read_csv("./data/일자별 행동 데이터.csv")

In [38]:
dailybehavior.head()

,user_id,date,device,visits,button_clicks_monthly,button_clicks_yearly
0,U2501,2026-01-15,PC,1,0,1
1,U2501,2026-01-16,PC,2,0,0
2,U2501,2026-01-17,PC,3,0,1
3,U2501,2026-01-18,PC,5,1,4
4,U2501,2026-01-20,PC,4,0,2


In [39]:
dailybehavior.isna().sum()

user_id                  0
date                     0
device                   0
visits                   0
button_clicks_monthly    0
button_clicks_yearly     0
dtype: int64

In [40]:
dailybehavior.duplicated().sum()

np.int64(0)